In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astroquery.simbad import Simbad
import warnings
warnings.filterwarnings('ignore')

print("Loading groups...")
groups = pd.read_pickle("/home/msp25gd/Downloads/res/meta/groups.pkl")
print(f"Loaded {len(groups)} groups")
print(f"\nGroups columns: {groups.columns.tolist()}")
print(f"\nFirst few rows:")
print(groups.head())

Loading groups...
Loaded 12827 groups

Groups columns: ['New Groups', 'OBJECT', 'Sanitised', 'Reduced', 'DEC', 'EXPTIME', 'MJD-OBS', 'MJD-END', 'WAVELMIN', 'WAVELMAX', 'SPEC_BIN', 'SPEC_RES']

First few rows:
   New Groups          OBJECT       Sanitised        Reduced       DEC  \
0           0    $\alpha$-Cru    $\alpha$ Cru    $\alpha$cru -63.09899   
1           1  $\gamma^2$-Vel  $\gamma^2$ Vel  $\gamma^2$vel -47.33665   
2           2    $\theta$-Car    $\theta$ Car    $\theta$car -64.39407   
3           3     $\zeta$-Pup     $\zeta$ Pup     $\zeta$pup -40.00284   
4           4       0003+1713       0003+1713      0003+1713  17.22642   

     EXPTIME       MJD-OBS       MJD-END    WAVELMIN    WAVELMAX  SPEC_BIN  \
0     5.0078  53747.372964  53747.375327  328.195459  456.300129  0.001358   
1     5.0079  53670.279645  53670.281882  328.192145  456.299040  0.001357   
2    10.0084  53686.311039  53686.313337  328.196918  456.298107  0.001356   
3     5.0081  53670.273791  53670.

In [ ]:
import re

# Load the quicksearch candidates (reduced names)
qs_candidates = np.load(
    '/home/msp25gd/ResearchProjectMSc/HR/results/QuickSearch/candidates_-4sig_2cut_3width.npy',
    allow_pickle=True
)
print(f"Quicksearch candidates: {len(qs_candidates)}")
print(qs_candidates[:10])

# Look up each candidate in groups.pkl using the Reduced column
# OBJECT is the raw FITS header name — same name iplotter passes to Simbad
metadata = pd.read_pickle("/home/msp25gd/Downloads/res/meta/metadata.pkl")

cand_rows = []
for reduced_name in qs_candidates:
    match = groups[groups['Reduced'] == reduced_name]
    if len(match) > 0:
        cand_rows.append(match.iloc[0])
    else:
        # fall back to metadata if the group lookup misses
        meta_match = metadata[metadata['Reduced'] == reduced_name]
        if len(meta_match) > 0:
            row = meta_match.iloc[0].copy()
            cand_rows.append(row)

candidates_df = pd.DataFrame(cand_rows).reset_index(drop=True)
print(f"\nResolved {len(candidates_df)} / {len(qs_candidates)} candidates")
print(f"Columns: {candidates_df.columns.tolist()}")
print(candidates_df[['OBJECT', 'Sanitised', 'Reduced']].head(10))


Querying Simbad for spectral types (this may take a few minutes)...
Querying 12827 targets...
Progress: 0/12827 (0.0%)
Progress: 500/12827 (3.9%)
Progress: 1000/12827 (7.8%)
Progress: 1500/12827 (11.7%)
Progress: 2000/12827 (15.6%)
Progress: 2500/12827 (19.5%)
Progress: 3000/12827 (23.4%)
Progress: 3500/12827 (27.3%)
Progress: 4000/12827 (31.2%)
Progress: 4500/12827 (35.1%)
Progress: 5000/12827 (39.0%)
Progress: 5500/12827 (42.9%)
Progress: 6000/12827 (46.8%)
Progress: 6500/12827 (50.7%)
Progress: 7000/12827 (54.6%)
Progress: 7500/12827 (58.5%)
Progress: 8000/12827 (62.4%)
Progress: 8500/12827 (66.3%)
Progress: 9000/12827 (70.2%)
Progress: 9500/12827 (74.1%)
Progress: 10000/12827 (78.0%)
Progress: 10500/12827 (81.9%)
Progress: 11000/12827 (85.8%)
Progress: 11500/12827 (89.7%)
Progress: 12000/12827 (93.6%)
Progress: 12500/12827 (97.5%)

Query complete!
Successfully retrieved spectral types: 0
Failed queries: 12827

Saved groups with spectral types to: /home/msp25gd/Downloads/res/meta/g

In [ ]:
# Query Simbad for spectral types using the OBJECT name
# (same approach as iplotter's get_otypes — this name is what Simbad knows)
from astroquery.simbad import Simbad

simbad = Simbad()
simbad.add_votable_fields('sp_type')
simbad.TIMEOUT = 120

print(f"Querying Simbad for {len(candidates_df)} candidates...\n")

sp_types = []
for idx, row in candidates_df.iterrows():
    object_name = row['OBJECT']
    try:
        result = simbad.query_object(object_name)
        if result is not None and len(result) > 0 and 'sp_type' in result.colnames:
            sp_type = str(result['sp_type'][0]).strip()
            sp_types.append(sp_type if sp_type else None)
        else:
            sp_types.append(None)
    except Exception:
        sp_types.append(None)
    print(f"  {idx+1:>3}/{len(candidates_df)}  {object_name:<30}  {sp_types[-1]}")

candidates_df['sp_type'] = sp_types

found = sum(1 for x in sp_types if x)
print(f"\nSpectral types found: {found} / {len(candidates_df)}")

Extracting spectral classes from query results...

Spectral Class Distribution:
A:   689
B:   903
C:    47
D:   417
F:   630
G:   758
K:   488
M:   208
O:   208
Other:    14
S:   130
Unknown:  8303
W:    32

Total with known spectral class: 4524
Unknown: 8303

Histogram saved to: /home/msp25gd/ResearchProjectMSc/stats/spectral_type_histogram.png


Extracting spectral classes from query results...

Spectral Class Distribution:
A:   689
B:   903
C:    47
D:   417
F:   630
G:   758
K:   488
M:   208
O:   208
Other:    14
S:   130
Unknown:  8303
W:    32

Total with known spectral class: 4524
Unknown: 8303

Histogram saved to: /home/msp25gd/ResearchProjectMSc/stats/spectral_type_histogram.png



SPECTRAL TYPE DISTRIBUTION SUMMARY
Class    Count    Percentage   Description
----------------------------------------------------------------------
O        208         4.6%     O-type (hottest, most massive)
B        903        20.0%     B-type (hot, blue)
A        689        15.2%     A-type (hot, blue-white)
F        630        13.9%     F-type (warm, white)
G        758        16.8%     G-type (like our Sun, yellow)
K        488        10.8%     K-type (warm, orange)
M        208         4.6%     M-type (coolest, red dwarfs)
W        32          0.7%     White dwarfs
C        47          1.0%     Carbon stars
S        130         2.9%     S-type (zirconium-rich giants)
D        417         9.2%     Dunham class (planetary nebula, etc)
Other    14          0.3%     Other types
Unknown  8303     N/A          Unknown/Not found in Simbad
----------------------------------------------------------------------
TOTAL (known) 4524     100.0%      
UNKNOWN  8303    


In [ ]:
import matplotlib.pyplot as plt

# Extract spectral class (first letter) from full spectral type string
CLASS_ORDER = ['O', 'B', 'A', 'F', 'G', 'K', 'M', 'W', 'C', 'S', 'Other', 'Unknown']
CLASS_COLORS = {
    'O': '#3355FF', 'B': '#6699FF', 'A': '#AADDFF',
    'F': '#CCFFEE', 'G': '#FFFF44', 'K': '#FF9900',
    'M': '#FF3333', 'W': '#FF99FF', 'C': '#8B5A2B',
    'S': '#FF6699', 'Other': '#BBBBBB', 'Unknown': '#888888'
}

def to_class(sp_type):
    if not sp_type or str(sp_type).strip() in ('', 'None', 'nan', '--'):
        return 'Unknown'
    letter = sp_type[0].upper()
    return letter if letter in list('OBAFGKMWCS') else 'Other'

candidates_df['sp_class'] = candidates_df['sp_type'].apply(to_class)

counts = candidates_df['sp_class'].value_counts()
# Preserve canonical order, dropping absent classes
present = [c for c in CLASS_ORDER if c in counts]
values  = [counts[c] for c in present]
colors  = [CLASS_COLORS[c] for c in present]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(present, values, color=colors, edgecolor='black', linewidth=1.2)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Spectral Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Stars', fontsize=12, fontweight='bold')
ax.set_title(
    f'Spectral Type Distribution of QuickSearch Candidates\n'
    f'({len(candidates_df)} candidates from {len(qs_candidates)} total)',
    fontsize=13, fontweight='bold', pad=15
)
ax.grid(axis='y', alpha=0.35, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('/home/msp25gd/ResearchProjectMSc/stats/spectral_type_quicksearch.png',
            dpi=300, bbox_inches='tight')
plt.show()

# Summary table
known_total = sum(v for c, v in zip(present, values) if c not in ('Unknown', 'Other'))
print(f"\n{'Class':<8} {'Count':<8} {'%'}")
print('-' * 28)
for cls, val in zip(present, values):
    pct = f'{100*val/known_total:.1f}%' if cls not in ('Unknown', 'Other') else 'N/A'
    print(f"{cls:<8} {val:<8} {pct}")